In [ ]:
# ============================================================
# ARTIGO 3 — P e K: Validação de cenário de manejo
# Comparação IDW x Krigagem com estatísticas, gráficos e discussão
# ============================================================

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# ============ 1) Carregar CSV ============
from google.colab import files
uploaded = files.upload()

df = pd.read_csv(list(uploaded.keys())[0], encoding='latin1', sep=';')

# Padronizar texto
df.columns = df.columns.str.strip()
df["Tipo"] = df["Tipo"].str.strip().str.upper()
df["Critério"] = df["Critério"].str.strip()

# Remover símbolos
for col in ["R²", "RMSE", "CVRMSE"]:
    df[col] = df[col].astype(str).str.replace("%", "").str.replace(",", ".").astype(float)

print("\n=== Dados carregados ===")
display(df)

# ============ 2) Estatísticas descritivas ============
estat = df.groupby(["Tipo", "Critério"]).agg(["count","mean","std","min","max"])
print("\n=== Estatísticas descritivas ===")
print(estat)

# ============ 3) Tabela média para discussão ============
tabela_media = df.groupby(["Tipo","Critério"])[["R²","RMSE","CVRMSE"]].mean()
print("\n=== Tabela média para discussão ===")
print(tabela_media)

# ============ 4) Gráficos comparativos ============
metricas = ["R²","RMSE","CVRMSE"]

for met in metricas:
    plt.figure(figsize=(10,5))
    sns.barplot(data=df, x="Critério", y=met, hue="Tipo", palette="viridis")
    plt.title(f"Comparação {met} — IDW vs Krigagem (P e K)")
    plt.xlabel("Nutriente")
    plt.ylabel(met)
    plt.legend(title="Método")
    plt.show()

print("\nGráficos gerados com sucesso!")

# ============ 5) Discussões automatizadas ============
def discutir(c):
    dfc = tabela_media.loc[pd.IndexSlice[:, c], :]
    melhor_rmse = dfc["RMSE"].idxmin()[0]
    rmse_val = dfc["RMSE"].min()
    return f"{melhor_rmse} apresentou menor RMSE para {c} ({rmse_val:.4f})."

print("\n=== Discussão automática ===")
print("P:", discutir("P"))
print("K:", discutir("K"))

print("\nConclusão geral:")
print("""
Os nutrientes P e K são fundamentais para validar o cenário de manejo.
A comparação IDW × Krigagem permite identificar o interpolador que melhor
representa a distribuição espacial, especialmente importante para zonas de manejo.
""")


In [ ]:
# =====================================
# GRÁFICOS COMPARATIVOS DO ARTIGO 3
# R², RMSE, CVRMSE por Nutriente
# =====================================

import seaborn as sns
import matplotlib.pyplot as plt

df = df  # seu DataFrame já carregado com todas as métricas

metricas = ["R²", "RMSE", "CVRMSE"]
variaveis = df["Critério"].unique()

for var in variaveis:
    subset = df[df["Critério"] == var]

    for metrica in metricas:
        plt.figure(figsize=(8,5))
        sns.barplot(
            data=subset,
            x="Amostragem",
            y=metrica,
            hue="Tipo",
            palette="viridis"
        )
        plt.title(f"{var} — {metrica} (IDW × Krigagem)")
        plt.xlabel("Amostragem (%)")
        plt.ylabel(metrica)
        plt.legend(title="Método")
        plt.tight_layout()
        plt.show()

print("\nGráficos comparativos gerados com sucesso!")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load CSV
df = pd.read_csv("/content/Interpolação.csv", encoding="latin1", sep=";")
df.columns = df.columns.str.replace("ï»¿", "")

# Convert coordinates
df["X_UTM"] = df["X_UTM"].astype(str).str.replace(",", ".").astype(float)
df["Y_UTM"] = df["Y_UTM"].astype(str).str.replace(",", ".").astype(float)

# Variables
variables = {
    "P_resina_mg_dm": "P (mg/dm³)",
    "K_resina_cmol_dm": "K (cmol/dm³)"
}

# IDW interpolation
def idw_interpolation(x, y, z, xi, yi, power=2):
    dist = np.sqrt((xi - x[:, None, None])**2 + (yi - y[:, None, None])**2)
    dist[dist == 0] = 1e-10
    weights = 1 / dist**power
    return np.sum(weights * z[:, None, None], axis=0) / np.sum(weights, axis=0)

# Coordinates
x = df["X_UTM"].values
y = df["Y_UTM"].values

xi = np.linspace(min(x), max(x), 150)
yi = np.linspace(min(y), max(y), 150)
Xi, Yi = np.meshgrid(xi, yi)

generated_files = []

for col, label in variables.items():
    df[col] = df[col].astype(str).str.replace(",", ".").astype(float)
    z = df[col].values

    Zi = idw_interpolation(x, y, z, Xi, Yi)

    plt.figure(figsize=(8,6))
    plt.contourf(Xi, Yi, Zi, 20)
    plt.scatter(x, y, c=z)
    plt.colorbar(label=label)
    plt.xlabel("X (UTM)")
    plt.ylabel("Y (UTM)")
    plt.title(f"Mapa Espacial – {label}")

    out_path = f"/content/Mapa_{col}.png" # Caminho de saída corrigido para /content/
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()

    generated_files.append(out_path)

generated_files

In [ ]:
from IPython.display import Image, display

print("=== Mapas Gerados ===")
for file_path in generated_files:
    print(f"Exibindo: {file_path}")
    display(Image(filename=file_path))


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 5))
plt.scatter(df["P_resina_mg_dm"], df["K_resina_cmol_dm"])
plt.xlabel("P (mg/dm³)")
plt.ylabel("K (cmol/dm³)")
plt.title("Relação entre P e K nos pontos amostrados")
plt.grid(True)
plt.tight_layout()
plt.savefig("Comparativo_P_vs_K.png", dpi=300)
plt.show() # Adicionado para exibir o gráfico
# plt.close() # Removido para permitir a exibição do gráfico

print("Gráfico comparativo P vs K gerado!")

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

# ============================================================
# 1) CARREGAR ARQUIVO COM OS DADOS DE SOLO (Interpolação.csv)
# ============================================================

uploaded_interpolacao = files.upload()
df = pd.read_csv(list(uploaded_interpolacao.keys())[0], sep=';', encoding='latin1')
df.columns = df.columns.str.replace('ï»¿','')

# Converter valores numéricos
for col in ["P_resina_mg_dm", "K_resina_cmol_dm"]:
    df[col] = df[col].astype(str).str.replace(',', '.').astype(float)

# ============================================================
# 2) DEFINIR A PRODUTIVIDADE ESPERADA (t/ha)
# ============================================================

produtividade = 120   # <<=== você pode alterar (ex: 80, 120, 160)

# ============================================================
# 3) TABELAS DE RECOMENDAÇÃO (você forneceu)
# ============================================================

# ======= FÓSFORO (P2O5 kg/ha) ========
# recomendação oficial: sempre 30 se P ≤ 15 mg/dm3
def recomendacao_P(p_resina):
    if p_resina <= 15:
        return 30   # kg/ha de P2O5
    else:
        return 0

# ======= POTÁSSIO (K2O kg/ha) ========
# tabela oficial por faixa de produtividade:

def recomendacao_K(k_cmol):

    # classes de K
    if k_cmol <= 0.7:
        classe = 1
    elif k_cmol <= 1.5:
        classe = 2
    elif k_cmol <= 3.0:
        classe = 3
    elif k_cmol <= 6.0:
        classe = 4
    else:
        classe = 5

    # agora aplicar produtividade
    if produtividade < 100:
        recs = {1:100, 2:80, 3:40, 4:40, 5:0}
    elif produtividade <= 150:
        recs = {1:150, 2:120, 3:80, 4:60, 5:0}
    else:
        recs = {1:200, 2:160, 3:120, 4:80, 5:0}

    return recs[classe]

# ============================================================
# 4) CALCULAR RECOMENDAÇÕES PONTO A PONTO
# ============================================================

df["P_recomendado"] = df["P_resina_mg_dm"].apply(recomendacao_P)
df["K_recomendado"] = df["K_resina_cmol_dm"].apply(recomendacao_K)

# criar mapas binários
df["P_bin"] = (df["P_recomendado"] > 0).astype(int)
df["K_bin"] = (df["K_recomendado"] > 0).astype(int)

print("Amostra das recomendações calculadas:")
print(df[["P_resina_mg_dm","P_recomendado","P_bin",
          "K_resina_cmol_dm","K_recomendado","K_bin"]].head())

# ============================================================
# 5) CALCULAR ÁREA (se você tiver área por ponto)
# ============================================================

# se cada ponto representar 1 ha:
ha_por_ponto = 1

df["Area_P_adubar"] = df["P_bin"] * ha_por_ponto
df["Area_K_adubar"] = df["K_bin"] * ha_por_ponto

print("\nÁrea total a adubar (P):", df["Area_P_adubar"].sum(), "ha")
print("Área total a adubar (K):", df["Area_K_adubar"].sum(), "ha")

# ============================================================
# 6) CUSTO DE ADUBAÇÃO (R$/ha)
# ============================================================

# preços (exemplo — você pode ajustar)
preco_P2O5 = 6.00   # R$/kg
preco_K2O  = 5.50   # R$/kg

df["Custo_P"] = df["P_recomendado"] * preco_P2O5
df["Custo_K"] = df["K_recomendado"] * preco_K2O

df["Custo_total"] = df["Custo_P"] + df["Custo_K"]

print("\nCusto total estimado (todos os pontos): R$ %.2f" % df["Custo_total"].sum())

df.to_csv("Recomendacoes_Finais.csv", index=False)
print("\nArquivo 'Recomendacoes_Finais.csv' gerado com sucesso!")